# Aula 02 — MCP II  
## Encapsulando um Workflow RAG em um Servidor MCP Local

**Banco BV — AI Experts | Ecossistema Moderno de Agentes**

Nesta aula, vamos dar um passo além da primeira aula de MCP.

Na **MCP I**, você viu como um agente pode se conectar a um servidor MCP, listar ferramentas e chamar funções externas. Agora vamos transformar um workflow inteiro de IA em uma ferramenta MCP reutilizável.

A ideia central é simples, mas poderosa:

> Um agente não precisa saber fazer tudo.  
> Ele precisa saber chamar a capacidade especialista correta.

Nesta aula, vamos construir um RAG sobre um regulamento financeiro, encapsular esse RAG em um servidor MCP local e consumir essa capacidade por meio de um agente mais simples.


## Mapa da aula

```text
1. Construir o RAG no notebook
   PDF -> páginas -> chunks -> embeddings -> ChromaDB -> retriever -> LLM

2. Transformar o RAG em um módulo reutilizável
   rag_core.py

3. Expor a capacidade especialista via MCP
   servidor_mcp_rag_fundos.py

4. Consumir o servidor MCP
   MultiServerMCPClient + load_mcp_tools

5. Criar um agente simples
   Agente orquestrador -> ferramenta MCP -> workflow RAG especialista
```

A aula é propositalmente feita no formato de notebook, com markdowns e células executáveis.  
Mesmo quando gerarmos arquivos `.py`, faremos isso dentro do próprio notebook para deixar o processo transparente.


## Por que isso importa?

Em protótipos, é comum colocar tudo dentro de um único agente:

```text
prompt gigante
+ leitura de documentos
+ busca vetorial
+ lógica de negócio
+ regras de resposta
+ integrações externas
```

Isso funciona no início, mas fica difícil de manter, testar e auditar.

A proposta desta aula é separar responsabilidades:

```text
Agente simples
    ↓ chama via MCP
Servidor MCP especialista
    ↓ executa
Workflow RAG documental
```

Esse padrão aproxima agentes de uma arquitetura corporativa:  
capacidades especialistas, contratos explícitos, versionamento, rastreabilidade e reutilização.


## Arquitetura final

```text
┌──────────────────────────────────────────────────────────────┐
│                    Agente Simples BV                         │
│                                                              │
│  - Recebe a pergunta do usuário                              │
│  - Decide usar uma ferramenta documental                     │
│  - Não conhece PDF, chunking, embeddings ou Chroma            │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               │ MCP / JSON-RPC
                               v
┌──────────────────────────────────────────────────────────────┐
│          Servidor MCP — Inteligência Documental               │
│                                                              │
│  Tool: consultar_regulamento(pergunta, k)                    │
│  Tool: buscar_trechos_regulamento(pergunta, k)               │
│  Tool: extrair_taxas_regulamento()                           │
│  Tool: status_rag()                                          │
└──────────────────────────────┬───────────────────────────────┘
                               │
                               v
┌──────────────────────────────────────────────────────────────┐
│                         Pipeline RAG                         │
│                                                              │
│  PDF -> chunks -> embeddings -> vector store -> retriever     │
│      -> prompt com contexto -> LLM -> resposta com fontes     │
└──────────────────────────────────────────────────────────────┘
```


## Objetivos de aprendizagem

Ao final da aula, você será capaz de:

1. Construir um RAG simples sobre um documento financeiro.
2. Persistir um índice vetorial local com ChromaDB.
3. Transformar um workflow especialista em um módulo Python reutilizável.
4. Expor esse workflow como ferramentas de um servidor MCP local.
5. Consumir ferramentas MCP em LangChain.
6. Criar um agente simples que usa uma capacidade documental sem conhecer sua implementação interna.


## Parte 0 — Setup do ambiente

Esta aula usa:

| Componente | Papel |
|---|---|
| `LangChain` | composição do RAG |
| `LangChain OpenAI` | LLM e embeddings |
| `PyPDF` | leitura do PDF |
| `ChromaDB` | vector store local |
| `MCP SDK` | criação do servidor MCP |
| `langchain-mcp-adapters` | ponte entre MCP e LangChain |
| `LangGraph` | agente simples consumidor das tools |
| `Pydantic` | resposta estruturada e validável |

> Observação: execute a instalação uma vez. Em alguns ambientes, pode ser necessário reiniciar o kernel após instalar pacotes.


In [ ]:
# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ============================================================
# Execute esta célula uma vez.
# Se o ambiente pedir restart do kernel, reinicie e continue a partir da próxima célula.

%pip install -U langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters --quiet
%pip install -U chromadb pypdf pydantic python-dotenv tiktoken --quiet
%pip install -U mcp langchain-mcp-adapters httpx httpx-sse langgraph --quiet


In [ ]:
# ============================================================
# IMPORTS BÁSICOS E DIAGNÓSTICO DO AMBIENTE
# ============================================================

import os
import sys
import json
import time
import shutil
import socket
import pathlib
import textwrap
import subprocess
import importlib.metadata
from pprint import pprint

from dotenv import load_dotenv

load_dotenv()

def versao_pacote(nome: str) -> str:
    try:
        return importlib.metadata.version(nome)
    except importlib.metadata.PackageNotFoundError:
        return "não instalado"

print("Ambiente pronto.")
print(f"Python                 : {sys.version.split()[0]}")
print(f"langchain             : {versao_pacote('langchain')}")
print(f"langchain-openai      : {versao_pacote('langchain-openai')}")
print(f"langchain-mcp-adapters: {versao_pacote('langchain-mcp-adapters')}")
print(f"mcp                   : {versao_pacote('mcp')}")
print(f"chromadb              : {versao_pacote('chromadb')}")
print(f"langgraph             : {versao_pacote('langgraph')}")

if not os.getenv("OPENAI_API_KEY"):
    print()
    print("ATENÇÃO: OPENAI_API_KEY não encontrada no ambiente.")
    print("Crie um arquivo .env com:")
    print("OPENAI_API_KEY=sua-chave-aqui")
else:
    print()
    print("OPENAI_API_KEY encontrada.")


## Parte 1 — Preparando os arquivos da aula

A aula espera o seguinte arquivo:

```text
data/KNCR_Regulamento_05-2025.pdf
```

Você pode trocar por outro PDF, desde que ajuste a variável `PDF_PATH`.

Para fins didáticos, vamos usar o regulamento do KNCR11 como documento de demonstração.  
O ponto principal não é o fundo em si, mas o padrão arquitetural:

```text
Documento financeiro -> RAG especialista -> servidor MCP -> agente consumidor
```


In [ ]:
# ============================================================
# ESTRUTURA DE DIRETÓRIOS
# ============================================================

BASE_DIR = pathlib.Path.cwd()
DATA_DIR = BASE_DIR / "data"
VECTOR_DIR = BASE_DIR / "vectorstore" / "chroma_kncr"
LOG_DIR = BASE_DIR / "logs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DIR.parent.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = DATA_DIR / "KNCR_Regulamento_05-2025.pdf"

print("Diretórios da aula:")
print(f"BASE_DIR  : {BASE_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"VECTOR_DIR: {VECTOR_DIR}")
print(f"LOG_DIR   : {LOG_DIR}")
print()
print(f"PDF esperado: {PDF_PATH}")

if not PDF_PATH.exists():
    print()
    print("PDF ainda não encontrado.")
    print("Coloque o arquivo KNCR_Regulamento_05-2025.pdf dentro da pasta data/")
    print("ou ajuste a variável PDF_PATH para apontar para outro regulamento.")
else:
    print("PDF encontrado.")


## Parte 2 — Construindo o RAG no notebook

Antes de encapsular qualquer coisa em MCP, precisamos construir e entender a capacidade especialista.

Um RAG possui duas fases conceituais:

```text
Indexação:
PDF -> texto -> chunks -> embeddings -> vector store

Consulta:
pergunta -> busca semântica -> contexto -> LLM -> resposta
```

Vamos fazer isso primeiro no notebook.


### 2.1 Carregando o PDF

Usaremos `PyPDFLoader` para extrair o texto página a página.

Cada página vira um objeto `Document` com:

- `page_content`: texto extraído
- `metadata`: informações como fonte e número da página


In [ ]:
# ============================================================
# CARREGAMENTO DO PDF
# ============================================================

from langchain_community.document_loaders import PyPDFLoader

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {PDF_PATH}\n"
        "Coloque o PDF na pasta data/ ou ajuste a variável PDF_PATH."
    )

loader = PyPDFLoader(str(PDF_PATH))
paginas = loader.load()

print("PDF carregado com sucesso.")
print(f"Total de páginas: {len(paginas)}")
print()
print("Preview da primeira página:")
print("-" * 80)
print(paginas[0].page_content[:1000])
print("-" * 80)
print()
print("Metadados da primeira página:")
pprint(paginas[0].metadata)


### Exercício rápido

Explore algumas páginas do documento.

Sugestões:

```python
print(paginas[5].page_content[:1500])
print(paginas[10].metadata)
```

Procure por termos como:

- taxa
- remuneração
- política de investimento
- cotistas
- gestor
- administrador


In [ ]:
# ============================================================
# EXERCÍCIO 1 — EXPLORAÇÃO DO DOCUMENTO
# ============================================================
# Altere os parâmetros abaixo para explorar o PDF.

pagina_para_ver = 0
quantidade_caracteres = 1500

print(f"Página selecionada: {pagina_para_ver}")
print("-" * 80)
print(paginas[pagina_para_ver].page_content[:quantidade_caracteres])
print("-" * 80)
print()
print("Metadados:")
pprint(paginas[pagina_para_ver].metadata)


### 2.2 Chunking

LLMs têm limite de contexto. Além disso, buscar em um documento inteiro costuma ser menos preciso.

Por isso, dividimos o texto em chunks.

A escolha de `chunk_size` e `chunk_overlap` afeta diretamente a qualidade do RAG:

| Parâmetro | Efeito |
|---|---|
| `chunk_size` pequeno | busca mais granular, mas pode perder contexto |
| `chunk_size` grande | mais contexto, mas busca menos precisa |
| `chunk_overlap` | reduz o risco de cortar informação importante no meio |


In [ ]:
# ============================================================
# CHUNKING DO DOCUMENTO
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(paginas)

print("Chunking realizado.")
print(f"Páginas originais: {len(paginas)}")
print(f"Chunks gerados   : {len(chunks)}")
print()
print("Exemplo de chunk:")
print("-" * 80)
print(chunks[min(10, len(chunks)-1)].page_content[:1000])
print("-" * 80)
print()
print("Metadados do chunk:")
pprint(chunks[min(10, len(chunks)-1)].metadata)


### 2.3 Criando embeddings e vector store

Embeddings transformam texto em vetores numéricos que representam significado semântico.

```text
"taxa de administração"
"remuneração do administrador"
"custo de gestão"

→ vetores próximos no espaço semântico
```

Vamos usar:

- `OpenAIEmbeddings`
- modelo `text-embedding-3-small`
- `ChromaDB` local como vector store


In [ ]:
# ============================================================
# CRIAÇÃO DO VECTOR STORE COM CHROMADB
# ============================================================

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrada. Defina a chave no .env antes de criar embeddings."
    )

EMBEDDING_MODEL = "text-embedding-3-small"
COLLECTION_NAME = "regulamento_kncr"

# Para fins didáticos, podemos recriar o índice.
# Em produção, você provavelmente não recriaria embeddings a cada execução.
RECRIAR_INDICE = False

if RECRIAR_INDICE and VECTOR_DIR.exists():
    shutil.rmtree(VECTOR_DIR)

embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_DIR),
)

print("Vector store criado/persistido.")
print(f"Diretório       : {VECTOR_DIR}")
print(f"Collection      : {COLLECTION_NAME}")
print(f"Embedding       : {EMBEDDING_MODEL}")
print(f"Chunks indexados: {len(chunks)}")


### 2.4 Testando busca semântica

Antes de envolver LLM, vamos testar se a recuperação está trazendo trechos relevantes.

Isso é uma prática importante:  
muitos problemas de RAG não estão no modelo gerador, mas na recuperação.


In [ ]:
# ============================================================
# TESTE DE BUSCA SEMÂNTICA
# ============================================================

pergunta_teste = "Qual é a taxa de administração ou remuneração do fundo?"
docs_relevantes = vector_store.similarity_search(pergunta_teste, k=4)

print(f"Pergunta: {pergunta_teste}")
print()
print(f"Trechos recuperados: {len(docs_relevantes)}")

for i, doc in enumerate(docs_relevantes, start=1):
    pagina_zero_based = doc.metadata.get("page", "N/A")
    pagina_humana = pagina_zero_based + 1 if isinstance(pagina_zero_based, int) else pagina_zero_based
    fonte = doc.metadata.get("source", "documento")

    print()
    print("=" * 80)
    print(f"Chunk {i} | Fonte: {fonte} | Página: {pagina_humana}")
    print("-" * 80)
    print(doc.page_content[:900])


### 2.5 Construindo a chain RAG

Agora vamos juntar:

```text
pergunta
  ↓
retriever
  ↓
contexto formatado
  ↓
prompt
  ↓
LLM
  ↓
resposta
```

Regra de ouro para contexto financeiro e regulatório:

> O modelo só pode responder com base no contexto recuperado.  
> Se a resposta não estiver no contexto, ele deve admitir que não encontrou.


In [ ]:
# ============================================================
# CHAIN RAG
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

CHAT_MODEL = "gpt-4o-mini"

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

def formatar_documentos(docs):
    partes = []
    for doc in docs:
        pagina_zero_based = doc.metadata.get("page", "N/A")
        pagina_humana = pagina_zero_based + 1 if isinstance(pagina_zero_based, int) else pagina_zero_based
        fonte = doc.metadata.get("source", "documento")
        partes.append(
            f"[Fonte: {fonte} | Página: {pagina_humana}]\n{doc.page_content}"
        )
    return "\n\n---\n\n".join(partes)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Você é um especialista em análise de regulamentos financeiros.

Responda usando APENAS o contexto fornecido.
Se a resposta não estiver no contexto, diga claramente que não encontrou a informação no documento.
Sempre que possível, cite fonte, página e/ou seção.
Não invente números, cláusulas, taxas ou regras.

Contexto:
{context}
"""),
    ("human", "{question}")
])

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

rag_chain = (
    {"context": retriever | formatar_documentos, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Chain RAG configurada.")
print(f"LLM       : {CHAT_MODEL}")
print(f"Retriever: similarity search, k=5")


In [ ]:
# ============================================================
# TESTE DA CHAIN RAG
# ============================================================

pergunta = "Qual é a taxa de administração ou remuneração prevista no regulamento?"
resposta = rag_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)
print()
print("Resposta:")
print("-" * 80)
print(resposta)


## Parte 3 — Resposta estruturada com fontes

Para uma aula corporativa, não basta retornar uma string bonita.

Queremos uma resposta com:

```json
{
  "pergunta": "...",
  "resposta": "...",
  "fontes": [
    {
      "documento": "...",
      "pagina": 16,
      "trecho": "..."
    }
  ],
  "paginas_consultadas": [16, 17],
  "confianca": "alta",
  "observacao": "Resposta baseada apenas nos trechos recuperados."
}
```

Esse tipo de contrato é importante para rastreabilidade, auditoria e depuração.


In [ ]:
# ============================================================
# MODELOS PYDANTIC PARA RESPOSTA ESTRUTURADA
# ============================================================

from typing import Literal, Optional
from pydantic import BaseModel, Field

class FonteConsulta(BaseModel):
    documento: Optional[str] = Field(default=None, description="Nome/caminho do documento consultado")
    pagina: Optional[int] = Field(default=None, description="Página em numeração humana, começando em 1")
    trecho: str = Field(description="Trecho recuperado do documento")

class RespostaRAG(BaseModel):
    pergunta: str
    resposta: str
    fontes: list[FonteConsulta]
    paginas_consultadas: list[int]
    confianca: Literal["alta", "media", "baixa"]
    observacao: str

def consultar_regulamento_lab(pergunta: str, k: int = 5) -> dict:
    docs = vector_store.similarity_search(pergunta, k=k)
    contexto = formatar_documentos(docs)

    resposta_texto = (rag_prompt | llm | StrOutputParser()).invoke({
        "context": contexto,
        "question": pergunta,
    })

    fontes = []
    paginas = []

    for doc in docs:
        pagina_zero_based = doc.metadata.get("page")
        pagina_humana = pagina_zero_based + 1 if isinstance(pagina_zero_based, int) else None
        if pagina_humana is not None:
            paginas.append(pagina_humana)

        fontes.append(FonteConsulta(
            documento=doc.metadata.get("source"),
            pagina=pagina_humana,
            trecho=doc.page_content[:700],
        ))

    paginas_unicas = sorted(set(paginas))

    if len(docs) >= 4:
        confianca = "alta"
    elif len(docs) >= 2:
        confianca = "media"
    else:
        confianca = "baixa"

    payload = RespostaRAG(
        pergunta=pergunta,
        resposta=resposta_texto,
        fontes=fontes,
        paginas_consultadas=paginas_unicas,
        confianca=confianca,
        observacao="Resposta gerada apenas com base nos trechos recuperados pelo RAG.",
    )

    return payload.model_dump()

resultado_estruturado = consultar_regulamento_lab(
    "Qual é a política de investimento do fundo?",
    k=5,
)

print(json.dumps(resultado_estruturado, indent=2, ensure_ascii=False)[:5000])


## Parte 4 — Do notebook para um módulo reutilizável

Até aqui, o workflow funciona, mas está preso no notebook.

Agora vamos transformar essa capacidade em um módulo Python:

```text
rag_core.py
```

Esse arquivo vai conter uma classe:

```python
RAGRegulamentoFundos
```

Ela será responsável por:

1. Carregar o PDF.
2. Dividir em chunks.
3. Criar ou carregar o índice vetorial.
4. Fazer busca semântica.
5. Gerar resposta com LLM.
6. Retornar payload estruturado.

Repare no desenho arquitetural:

```text
Notebook = laboratório e material didático
rag_core.py = capacidade especialista reutilizável
servidor_mcp_rag_fundos.py = fronteira MCP
agente = consumidor simples
```


In [ ]:
%%writefile rag_core.py
"""
rag_core.py

Capacidade especialista de RAG para consulta de regulamentos financeiros.

Este módulo é usado pelo servidor MCP da Aula 02 - MCP II.

Arquitetura:
    PDF -> chunks -> embeddings -> ChromaDB -> retriever -> LLM -> resposta estruturada
"""

from __future__ import annotations

import os
import shutil
from pathlib import Path
from typing import Literal, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


class FonteConsulta(BaseModel):
    """Fonte textual recuperada pelo RAG."""

    documento: Optional[str] = Field(default=None, description="Nome/caminho do documento consultado")
    pagina: Optional[int] = Field(default=None, description="Página em numeração humana, começando em 1")
    trecho: str = Field(description="Trecho recuperado do documento")


class RespostaRAG(BaseModel):
    """Resposta estruturada devolvida pelo workflow RAG."""

    pergunta: str
    resposta: str
    fontes: list[FonteConsulta]
    paginas_consultadas: list[int]
    confianca: Literal["alta", "media", "baixa"]
    observacao: str


class RAGRegulamentoFundos:
    """
    Workflow especialista de RAG para regulamentos financeiros.

    Esta classe encapsula tudo que o agente consumidor NÃO precisa saber:
    carregamento de PDF, chunking, embeddings, vector store, prompt e LLM.

    O agente chamará apenas ferramentas MCP.
    O servidor MCP chamará esta classe.
    """

    def __init__(
        self,
        pdf_path: str = "data/KNCR_Regulamento_05-2025.pdf",
        persist_directory: str = "vectorstore/chroma_kncr",
        collection_name: str = "regulamento_kncr",
        embedding_model: str = "text-embedding-3-small",
        chat_model: str = "gpt-4o-mini",
        chunk_size: int = 800,
        chunk_overlap: int = 120,
        k: int = 5,
        recriar_indice: bool = False,
    ):
        load_dotenv()

        self.pdf_path = Path(pdf_path)
        self.persist_directory = Path(persist_directory)
        self.collection_name = collection_name
        self.embedding_model_name = embedding_model
        self.chat_model_name = chat_model
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.k = k
        self.recriar_indice = recriar_indice

        if not os.getenv("OPENAI_API_KEY"):
            raise EnvironmentError(
                "OPENAI_API_KEY não encontrada. Defina a variável no ambiente ou em um arquivo .env."
            )

        self.persist_directory.parent.mkdir(parents=True, exist_ok=True)

        self.embeddings = OpenAIEmbeddings(model=self.embedding_model_name)
        self.llm = ChatOpenAI(model=self.chat_model_name, temperature=0)

        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """
Você é um especialista em análise de regulamentos financeiros.

Responda usando APENAS o contexto fornecido.
Se a resposta não estiver no contexto, diga claramente que não encontrou a informação no documento.
Sempre que possível, cite fonte, página e/ou seção.
Não invente números, cláusulas, taxas ou regras.

Contexto:
{context}
"""),
            ("human", "{question}")
        ])

        self.vector_store = self._preparar_vector_store()

    def _preparar_vector_store(self) -> Chroma:
        """
        Cria ou carrega o índice vetorial.

        Em produção, o índice normalmente seria criado por um pipeline separado.
        Nesta aula, deixamos a criação dentro da classe para fins didáticos.
        """

        if self.recriar_indice and self.persist_directory.exists():
            shutil.rmtree(self.persist_directory)

        vector_store = Chroma(
            collection_name=self.collection_name,
            embedding_function=self.embeddings,
            persist_directory=str(self.persist_directory),
        )

        try:
            quantidade = vector_store._collection.count()
        except Exception:
            quantidade = 0

        if quantidade == 0:
            if not self.pdf_path.exists():
                raise FileNotFoundError(
                    f"PDF não encontrado: {self.pdf_path}. "
                    "Coloque o arquivo em data/ ou ajuste pdf_path."
                )

            paginas = self._carregar_pdf()
            chunks = self._dividir_documentos(paginas)

            vector_store = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                collection_name=self.collection_name,
                persist_directory=str(self.persist_directory),
            )

        return vector_store

    def _carregar_pdf(self):
        """Carrega o PDF em objetos Document do LangChain."""

        loader = PyPDFLoader(str(self.pdf_path))
        return loader.load()

    def _dividir_documentos(self, paginas):
        """Divide páginas em chunks menores para busca semântica."""

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        return splitter.split_documents(paginas)

    @staticmethod
    def _pagina_humana(doc) -> Optional[int]:
        """
        Converte a página zero-based do PyPDFLoader para numeração humana.

        PyPDFLoader costuma retornar page=0 para a primeira página.
        Para o usuário final, mostramos página 1.
        """

        pagina = doc.metadata.get("page")
        if isinstance(pagina, int):
            return pagina + 1
        return None

    def _formatar_documentos(self, docs) -> str:
        """Formata documentos recuperados para entrada no prompt."""

        partes = []
        for doc in docs:
            pagina = self._pagina_humana(doc)
            fonte = doc.metadata.get("source", "documento")
            partes.append(
                f"[Fonte: {fonte} | Página: {pagina}]\n{doc.page_content}"
            )
        return "\n\n---\n\n".join(partes)

    def _fontes(self, docs) -> tuple[list[FonteConsulta], list[int]]:
        """Extrai fontes e páginas a partir dos documentos recuperados."""

        fontes: list[FonteConsulta] = []
        paginas: list[int] = []

        for doc in docs:
            pagina = self._pagina_humana(doc)
            if pagina is not None:
                paginas.append(pagina)

            fontes.append(FonteConsulta(
                documento=doc.metadata.get("source"),
                pagina=pagina,
                trecho=doc.page_content[:700],
            ))

        return fontes, sorted(set(paginas))

    @staticmethod
    def _inferir_confianca(docs) -> Literal["alta", "media", "baixa"]:
        """
        Heurística didática de confiança.

        Em produção, esta métrica deveria ser substituída por avaliação mais robusta:
        scores do retriever, checagem de citação, avaliação automática, ou validação humana.
        """

        if len(docs) >= 4:
            return "alta"
        if len(docs) >= 2:
            return "media"
        return "baixa"

    def buscar_trechos(self, pergunta: str, k: Optional[int] = None) -> dict:
        """
        Retorna apenas os trechos mais relevantes, sem chamar o LLM.

        Útil para debug do retriever.
        """

        k_final = k or self.k
        docs = self.vector_store.similarity_search(pergunta, k=k_final)
        fontes, paginas = self._fontes(docs)

        return {
            "pergunta": pergunta,
            "k": k_final,
            "paginas_consultadas": paginas,
            "fontes": [fonte.model_dump() for fonte in fontes],
        }

    def consultar(self, pergunta: str, k: Optional[int] = None) -> dict:
        """
        Consulta o regulamento usando RAG e retorna resposta estruturada.

        Args:
            pergunta: pergunta em linguagem natural.
            k: quantidade de chunks recuperados.

        Returns:
            dict compatível com JSON, contendo resposta e fontes.
        """

        k_final = k or self.k
        docs = self.vector_store.similarity_search(pergunta, k=k_final)
        contexto = self._formatar_documentos(docs)

        resposta_texto = (self.prompt | self.llm | StrOutputParser()).invoke({
            "context": contexto,
            "question": pergunta,
        })

        fontes, paginas = self._fontes(docs)

        resposta = RespostaRAG(
            pergunta=pergunta,
            resposta=resposta_texto,
            fontes=fontes,
            paginas_consultadas=paginas,
            confianca=self._inferir_confianca(docs),
            observacao="Resposta gerada apenas com base nos trechos recuperados pelo RAG.",
        )

        return resposta.model_dump()

    def extrair_taxas(self) -> dict:
        """
        Consulta orientada para extração de taxas, remunerações e custos.

        Esta tool reaproveita o mesmo workflow, mas fixa uma pergunta especialista.
        """

        pergunta = (
            "Liste as taxas, remunerações e custos relevantes previstos no regulamento. "
            "Inclua percentuais, responsáveis pelo pagamento e páginas de referência quando possível."
        )
        return self.consultar(pergunta=pergunta, k=max(self.k, 6))

    def status(self) -> dict:
        """Retorna informações operacionais do índice e dos modelos usados."""

        try:
            quantidade_chunks = self.vector_store._collection.count()
        except Exception:
            quantidade_chunks = None

        return {
            "pdf_path": str(self.pdf_path),
            "pdf_existe": self.pdf_path.exists(),
            "persist_directory": str(self.persist_directory),
            "collection_name": self.collection_name,
            "embedding_model": self.embedding_model_name,
            "chat_model": self.chat_model_name,
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "k_default": self.k,
            "quantidade_chunks_indexados": quantidade_chunks,
        }


### Testando o módulo `rag_core.py`

Antes de criar o servidor MCP, vamos importar a classe e testar a capacidade especialista.

Isso é importante:  
se o RAG não funciona isoladamente, não adianta culpar o MCP.


In [ ]:
# ============================================================
# TESTE DO MÓDULO rag_core.py
# ============================================================

from rag_core import RAGRegulamentoFundos

rag_especialista = RAGRegulamentoFundos(
    pdf_path=str(PDF_PATH),
    persist_directory=str(VECTOR_DIR),
    collection_name=COLLECTION_NAME,
    embedding_model=EMBEDDING_MODEL,
    chat_model=CHAT_MODEL,
    k=5,
    recriar_indice=False,
)

print("Status do RAG especialista:")
pprint(rag_especialista.status())

print()
print("Consulta de teste:")
resultado = rag_especialista.consultar("Qual é a taxa de administração ou remuneração do fundo?", k=5)
print(json.dumps(resultado, indent=2, ensure_ascii=False)[:5000])


## Parte 5 — Criando o servidor MCP

Agora vem a virada da aula.

Vamos expor a capacidade especialista como um servidor MCP local.

O servidor terá quatro tools:

| Tool | Função |
|---|---|
| `consultar_regulamento` | executa o RAG completo e retorna resposta com fontes |
| `buscar_trechos_regulamento` | retorna apenas trechos recuperados, útil para debug |
| `extrair_taxas_regulamento` | consulta especialista pré-configurada para taxas |
| `status_rag` | mostra status operacional do índice |

Observe que o agente consumidor não precisa conhecer `PyPDFLoader`, `ChromaDB`, embeddings ou prompt.  
Ele só conhece as tools.


In [ ]:
%%writefile servidor_mcp_rag_fundos.py
"""
servidor_mcp_rag_fundos.py

Servidor MCP local que encapsula um workflow RAG sobre regulamentos financeiros.

Inicie com:
    python servidor_mcp_rag_fundos.py

Endpoint:
    http://127.0.0.1:8766/mcp
"""

from __future__ import annotations

from typing import Optional

from mcp.server.fastmcp import FastMCP

from rag_core import RAGRegulamentoFundos


mcp = FastMCP(
    "BVInteligenciaDocumentalMCP",
    host="127.0.0.1",
    port=8766,
)

_rag: Optional[RAGRegulamentoFundos] = None


def get_rag() -> RAGRegulamentoFundos:
    """
    Inicialização lazy do RAG.

    O servidor sobe rápido, e o índice é carregado/criado apenas quando uma tool precisa dele.
    """

    global _rag

    if _rag is None:
        _rag = RAGRegulamentoFundos(
            pdf_path="data/KNCR_Regulamento_05-2025.pdf",
            persist_directory="vectorstore/chroma_kncr",
            collection_name="regulamento_kncr",
            embedding_model="text-embedding-3-small",
            chat_model="gpt-4o-mini",
            k=5,
            recriar_indice=False,
        )

    return _rag


@mcp.tool()
def consultar_regulamento(pergunta: str, k: int = 5) -> dict:
    """
    Consulta o regulamento financeiro usando RAG.

    Use esta ferramenta quando a pergunta envolver:
    - regras do regulamento
    - política de investimento
    - taxas e remuneração
    - obrigações de administrador, gestor ou cotistas
    - informações que exigem fonte documental

    Args:
        pergunta: pergunta em linguagem natural sobre o regulamento.
        k: quantidade de trechos recuperados no vector store.

    Returns:
        Resposta estruturada com resposta, fontes, páginas e confiança.
    """

    return get_rag().consultar(pergunta=pergunta, k=k)


@mcp.tool()
def buscar_trechos_regulamento(pergunta: str, k: int = 5) -> dict:
    """
    Busca trechos relevantes no regulamento sem chamar o LLM.

    Esta ferramenta é útil para depurar a qualidade do retriever.

    Args:
        pergunta: consulta semântica.
        k: quantidade de trechos recuperados.

    Returns:
        Trechos recuperados, páginas e metadados.
    """

    return get_rag().buscar_trechos(pergunta=pergunta, k=k)


@mcp.tool()
def extrair_taxas_regulamento() -> dict:
    """
    Extrai taxas, remunerações e custos relevantes previstos no regulamento.

    Returns:
        Resposta estruturada com taxas encontradas e fontes.
    """

    return get_rag().extrair_taxas()


@mcp.tool()
def status_rag() -> dict:
    """
    Mostra o status operacional do RAG.

    Returns:
        Informações sobre PDF, índice vetorial, modelos e quantidade de chunks.
    """

    return get_rag().status()


if __name__ == "__main__":
    mcp.run(transport="streamable-http")


## Parte 6 — Criando `requirements.txt`

Mesmo em aula, vale ensinar o aluno a deixar o projeto minimamente reproduzível.

Vamos gerar um `requirements.txt` simples.


In [ ]:
%%writefile requirements.txt
langchain
langchain-openai
langchain-community
langchain-chroma
langchain-text-splitters
chromadb
pypdf
pydantic
python-dotenv
tiktoken
mcp
langchain-mcp-adapters
httpx
httpx-sse
langgraph


## Parte 7 — Iniciando o servidor MCP local

Vamos iniciar o servidor como subprocesso.

Fluxo:

```text
Notebook
   ↓ inicia subprocesso
python servidor_mcp_rag_fundos.py
   ↓ expõe
http://127.0.0.1:8766/mcp
```

Em ambientes Jupyter, subprocesso é uma forma prática de rodar servidor local sem travar o notebook.


In [ ]:
# ============================================================
# INICIANDO O SERVIDOR MCP LOCAL
# ============================================================

SERVER_PATH = BASE_DIR / "servidor_mcp_rag_fundos.py"
MCP_PORT = 8766
SERVER_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"

def porta_aberta(host: str, port: int, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

stdout_log = open(LOG_DIR / "mcp_rag_stdout.log", "w", encoding="utf-8")
stderr_log = open(LOG_DIR / "mcp_rag_stderr.log", "w", encoding="utf-8")

# Evita iniciar outro processo se a porta já estiver ocupada.
if porta_aberta("127.0.0.1", MCP_PORT):
    print(f"A porta {MCP_PORT} já está aberta. Provavelmente o servidor já está rodando.")
    servidor_proc = None
else:
    servidor_proc = subprocess.Popen(
        [sys.executable, str(SERVER_PATH)],
        stdout=stdout_log,
        stderr=stderr_log,
        cwd=str(BASE_DIR),
        text=True,
    )

    print(f"Servidor iniciado. PID: {servidor_proc.pid}")
    print(f"Aguardando porta {MCP_PORT}...")

    for tentativa in range(40):
        if porta_aberta("127.0.0.1", MCP_PORT):
            print("Servidor respondendo.")
            break
        time.sleep(0.5)
    else:
        print("Servidor não abriu a porta no tempo esperado.")
        print("Veja os logs em:")
        print(LOG_DIR / "mcp_rag_stdout.log")
        print(LOG_DIR / "mcp_rag_stderr.log")

print(f"URL MCP: {SERVER_URL}")


## Parte 8 — Conectando no servidor MCP

Agora vamos nos comportar como um cliente MCP.

Usaremos:

```python
MultiServerMCPClient
load_mcp_tools
```

O objetivo é transformar as tools do servidor MCP em tools compatíveis com LangChain.


In [ ]:
# ============================================================
# CONECTANDO AO SERVIDOR MCP E LISTANDO TOOLS
# ============================================================

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

mcp_config = {
    "ragFundos": {
        "url": SERVER_URL,
        "transport": "streamable_http",
    }
}

cliente_mcp = MultiServerMCPClient(mcp_config)

async with cliente_mcp.session("ragFundos") as session:
    ferramentas_mcp = await load_mcp_tools(session)

print(f"Ferramentas disponíveis: {len(ferramentas_mcp)}")
print()

for tool in ferramentas_mcp:
    print(f"- {tool.name}")
    descricao = tool.description or ""
    print(f"  {descricao[:300]}...")
    print()


## Parte 9 — Chamando a tool MCP diretamente

Antes de criar um agente, vamos chamar a ferramenta diretamente.

Isso ajuda a separar dois problemas:

```text
1. A tool MCP funciona?
2. O agente sabe escolher e usar a tool?
```

Se a chamada direta falha, o problema está no servidor, transporte ou RAG.  
Se a chamada direta funciona e o agente falha, o problema está na orquestração do agente.


In [ ]:
# ============================================================
# CHAMADA DIRETA DA TOOL consultar_regulamento
# ============================================================

async with cliente_mcp.session("ragFundos") as session:
    tools = await load_mcp_tools(session)
    consultar_tool = next(t for t in tools if t.name == "consultar_regulamento")

    resultado_mcp = await consultar_tool.ainvoke({
        "pergunta": "Qual é a taxa de administração ou remuneração prevista no regulamento?",
        "k": 5,
    })

print("Resultado MCP:")
print("-" * 80)

if isinstance(resultado_mcp, str):
    print(resultado_mcp[:5000])
else:
    print(json.dumps(resultado_mcp, indent=2, ensure_ascii=False)[:5000])


In [ ]:
# ============================================================
# CHAMADA DIRETA DA TOOL buscar_trechos_regulamento
# ============================================================

async with cliente_mcp.session("ragFundos") as session:
    tools = await load_mcp_tools(session)
    buscar_tool = next(t for t in tools if t.name == "buscar_trechos_regulamento")

    trechos = await buscar_tool.ainvoke({
        "pergunta": "política de investimento do fundo",
        "k": 3,
    })

print("Trechos recuperados:")
print("-" * 80)

if isinstance(trechos, str):
    print(trechos[:5000])
else:
    print(json.dumps(trechos, indent=2, ensure_ascii=False)[:5000])


## Parte 10 — Criando um agente simples consumidor

Agora criamos um agente cuja responsabilidade é bem menor.

Ele não sabe como o RAG funciona.  
Ele só sabe que existe uma ferramenta chamada `consultar_regulamento`.

Padrão arquitetural:

```text
Usuário pergunta
   ↓
Agente decide usar tool
   ↓
Tool MCP chama servidor local
   ↓
Servidor executa RAG
   ↓
Agente recebe resposta estruturada
   ↓
Agente responde ao usuário preservando fontes
```


In [ ]:
# ============================================================
# AGENTE SIMPLES COM LANGGRAPH
# ============================================================

from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

llm_agente = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_agente = """
Você é um assistente financeiro do Banco BV.

Sua função é responder perguntas sobre documentos financeiros e regulamentos.
Quando a pergunta envolver regras, taxas, políticas, cláusulas, obrigações ou informações documentais,
use as ferramentas MCP disponíveis.

Regras:
- Não invente informações.
- Preserve fontes, páginas e observações retornadas pelas ferramentas.
- Se a ferramenta indicar que a informação não foi encontrada, informe isso claramente.
- Responda em português brasileiro, com clareza e objetividade.
"""

async with cliente_mcp.session("ragFundos") as session:
    tools = await load_mcp_tools(session)

    agente = create_react_agent(
        model=llm_agente,
        tools=tools,
        prompt=prompt_agente,
    )

    resposta_agente = await agente.ainvoke({
        "messages": [
            ("user", "Qual é a política de investimento do fundo? Cite as páginas usadas.")
        ]
    })

print("Mensagens do agente:")
print("=" * 80)

for msg in resposta_agente["messages"]:
    try:
        msg.pretty_print()
    except Exception:
        print(msg)


### Teste adicional do agente

Faça perguntas como:

```text
1. Quais são as principais taxas do regulamento?
2. Quem é o administrador do fundo?
3. O regulamento fala sobre distribuição de rendimentos?
4. O que acontece se a informação não estiver no documento?
```


In [ ]:
# ============================================================
# EXERCÍCIO 2 — TESTE LIVRE DO AGENTE
# ============================================================

pergunta_usuario = "Liste as taxas relevantes do regulamento e cite as fontes."

async with cliente_mcp.session("ragFundos") as session:
    tools = await load_mcp_tools(session)

    agente = create_react_agent(
        model=llm_agente,
        tools=tools,
        prompt=prompt_agente,
    )

    resposta_agente = await agente.ainvoke({
        "messages": [
            ("user", pergunta_usuario)
        ]
    })

ultima_mensagem = resposta_agente["messages"][-1]
try:
    ultima_mensagem.pretty_print()
except Exception:
    print(ultima_mensagem)


## Parte 11 — O que acabamos de construir?

Criamos uma separação limpa:

```text
rag_core.py
    Implementação da capacidade especialista

servidor_mcp_rag_fundos.py
    Contrato MCP para expor essa capacidade

agente simples
    Consumidor da capacidade, sem conhecer detalhes internos
```

Isso é diferente de colocar um “agente dentro de outro agente”.

O padrão desta aula é:

```text
Workflow especialista -> Tool MCP -> Agente consumidor
```

Mais tarde, em arquiteturas multiagentes, poderíamos evoluir para:

```text
Agente supervisor -> MCP -> Agente especialista
```

Mas primeiro é melhor dominar o padrão limpo: capacidade especialista encapsulada como ferramenta.


## Parte 12 — Exercícios de evolução

### Exercício 1 — Ajuste de recuperação

Altere:

```python
chunk_size
chunk_overlap
k
```

Compare a qualidade das respostas.

Perguntas de teste:

```text
Qual é a política de investimento?
Qual é a taxa de administração?
Quem são os prestadores de serviço?
Quais obrigações aparecem no regulamento?
```

---

### Exercício 2 — Nova tool MCP

Adicione ao servidor:

```python
@mcp.tool()
def resumir_regulamento() -> dict:
    ...
```

Objetivo: gerar um resumo executivo do documento.

---

### Exercício 3 — Outra persona de consulta

Crie uma tool:

```python
@mcp.tool()
def responder_compliance(pergunta: str) -> dict:
    ...
```

Ela deve responder com tom mais conservador, enfatizando limitações, fontes e incertezas.

---

### Exercício 4 — Trocar o documento

Substitua o PDF por outro regulamento ou política interna fictícia.

Objetivo: perceber que o agente consumidor não muda.  
Quem muda é o servidor especialista e seu índice.


## Parte 13 — Discussão arquitetural

Esta aula não é apenas sobre RAG.

É sobre transformar um workflow de IA em uma capacidade reutilizável.

Em um banco, poderíamos ter vários servidores MCP internos:

```text
mcp-credito
mcp-compliance
mcp-documentos
mcp-risco
mcp-atendimento
mcp-produtos
```

Cada servidor encapsula uma capacidade com fronteira clara.

O agente deixa de ser um bloco monolítico e passa a ser um orquestrador de serviços especialistas.

Essa é a analogia importante:

```text
Monólito de agente:
    prompt enorme + lógica + documentos + ferramentas + regras

Ecossistema de agentes:
    agente orquestrador + servidores MCP especialistas
```

O resultado é mais testável, auditável e evolutivo.


## Parte 14 — Encerramento

Você viu nesta aula:

1. Como construir um RAG documental.
2. Como retornar respostas com fontes.
3. Como transformar o RAG em um módulo Python.
4. Como expor esse módulo em um servidor MCP local.
5. Como conectar um cliente MCP.
6. Como transformar tools MCP em tools LangChain.
7. Como criar um agente simples que consome uma capacidade especialista.

Frase final da aula:

> MCP permite transformar workflows de IA em capacidades plugáveis.  
> O agente deixa de ser um bloco monolítico e passa a operar como um orquestrador de serviços especialistas.


## Referências e leituras recomendadas

- Anthropic — Model Context Protocol.
- LangChain — documentação de RAG, retrievers, tools e agents.
- LangGraph — agentes e arquiteturas de orquestração.
- ChromaDB — vector store local para desenvolvimento.
- Sam Newman — *Building Microservices*, 2ª edição.
- Mayo Oshin e Nuno Campos — *Learning LangChain*, O'Reilly.
- Noah Gift e Alfredo Deza — *Practical MLOps*, O'Reilly.


## Limpeza opcional

Ao final da aula, se quiser encerrar o servidor MCP iniciado pelo notebook:

```python
if servidor_proc is not None:
    servidor_proc.terminate()
```

Em alguns sistemas, você também pode precisar encerrar o processo manualmente se reiniciar o kernel.


In [ ]:
# ============================================================
# ENCERRAR SERVIDOR MCP — OPCIONAL
# ============================================================
# Execute esta célula apenas quando terminar os testes da aula.

# if servidor_proc is not None:
#     servidor_proc.terminate()
#     print("Servidor MCP encerrado.")
# else:
#     print("Nenhum processo iniciado por este notebook para encerrar.")
